# Week 2 — Class 3: Data Orchestration with Pandas
## Interactive Lecture Notebook

**Run each cell sequentially** to see concepts in action.

---

# 1. DataFrames and Series

### Topic
Pandas' core data structures — the `DataFrame` (2D labeled table) and `Series` (1D labeled array).

### Why It Is Related
Before you train a model, you must load, clean, transform, and explore data. Pandas is the Swiss Army knife for this phase. Every AI pipeline starts with `pd.read_csv()` and ends with `df.to_numpy()` before feeding to PyTorch.

### How It Works
- **DataFrame**: Dictionary-like container of Series, aligned by shared index. Think 'Excel spreadsheet with superpowers.'
- **Series**: Single column of data with an index.
- **Index**: Row labels (can be integers, strings, dates, or multi-level).
- **Dtype**: Pandas infers types automatically but often gets it wrong.

---

### Analogy
🍳 **DataFrame = Restaurant Kitchen Prep Station**

- Each column (Series) = a different ingredient: onions, tomatoes, garlic.
- The index = prep order number: 'Onion batch #1, #2, #3.'
- You slice by order number, by ingredient, or by both.
- You combine stations (merge), count batches (groupby), check freshness (filter).

---

## 1.1 Creating DataFrames & Understanding Structure

**RUN THE CELL BELOW** 👇

In [5]:
import pandas as pd
import numpy as np

# Create a DataFrame from a dictionary
data = {
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005'],
    'age': [25, 34, 45, 28, 52],
    'income': [45000, 62000, 88000, 51000, 95000],
    'country': ['USA', 'UK', 'USA', 'Germany', 'UK'],
    'is_active': [True, True, False, True, False],
}


df = pd.DataFrame(data)

# print("DataFrame structure:")
# print(f"  Shape: {df.shape} (rows × columns)")
# print(f"  Columns: {list(df.columns)}")
# print(f"  Index: {list(df.index)}")
# print(f"  Dtypes:\n{df.dtypes}")

# print(f"\nMemory usage:")
# print(f"  Total: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
# print(df.memory_usage(deep=True))

# # WHY memory matters? At 1M rows × 100 columns of float64 = ~800MB.
# # Converting to float32 halves this. In AI, memory is your bottleneck.

# # Access a column (returns a Series)
ages = df['age']
print(f"\nSeries 'age':")
print(f"  Type: {type(ages)}")
print(f"  Values: {ages.values}")  # Underlying NumPy array
print(f"  Index: {list(ages.index)}")

# ANALOGY: DataFrame = prep station with multiple ingredient trays.
#          Series = one tray. .values = the raw ingredients (NumPy array).


Series 'age':
  Type: <class 'pandas.Series'>
  Values: [25 34 45 28 52]
  Index: [0, 1, 2, 3, 4]


## 1.2 Indexing & Slicing — Label vs. Position

**RUN THE CELL BELOW** 👇

In [12]:
df

,customer_id,age,income,country,is_active
0,C001,25,45000,USA,True
1,C002,34,62000,UK,True
2,C003,45,88000,USA,False
3,C004,28,51000,Germany,True
4,C005,52,95000,UK,False


In [13]:
# Set customer_id as index (like a database primary key)
df_indexed = df.set_index('customer_id')

# print("With customer_id as index:")
# # print(df_indexed)
# df_indexed

# .loc[] — label-based indexing
# print(f"\n.loc['C002'] (label-based):\n{df_indexed.loc['C002']}")

# .iloc[] — position-based indexing
print(f"\n.iloc[1] (position-based):\n{df_indexed.iloc[1]}")

# # Slice by label range
# print(f"\n.loc['C002':'C004'] (label slice):\n{df_indexed.loc['C002':'C004']}")

# # Boolean indexing (filtering)
# high_earners = df_indexed[df_indexed['income'] > 60000]
# print(f"\nHigh earners (>60k):\n{high_earners}")

# # Multiple conditions
# active_us = df_indexed[(df_indexed['is_active'] == True) & (df_indexed['country'] == 'USA')]
# print(f"\nActive US customers:\n{active_us}")

# WHY .loc vs .iloc? .loc uses labels (robust to row reordering).
# .iloc uses positions (fast but fragile).
# In production, always use .loc with meaningful indices.

# ANALOGY: .loc = 'find the folder labeled C002.'
#          .iloc = 'find the 2nd folder from the top.'
#          If someone reorders folders, .loc still works, .iloc breaks.


.iloc[1] (position-based):
age             34
income       62000
country         UK
is_active     True
Name: C002, dtype: object


## 1.3 Data Types & Memory Optimization

**RUN THE CELL BELOW** 👇

In [16]:
# Create a larger dataset to see memory impact
np.random.seed(42)
n = 100_000


large_df = pd.DataFrame({
    'id': range(n),
    'age': np.random.randint(18, 80, n),
    'income': np.random.randint(20000, 150000, n),
    'category': np.random.choice(['premium', 'standard', 'basic'], n),
    'is_active': np.random.choice([True, False], n),
})

# large_df

# print("Default memory usage:")
# print(f"  Total: {large_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
# print(large_df.dtypes)

# # Optimize: convert to smaller dtypes
large_df['id'] = large_df['id'].astype('int32')      # int64 → int32
large_df['age'] = large_df['age'].astype('int8')     # int64 → int8 (ages fit in 0-127)
large_df['income'] = large_df['income'].astype('int32')
large_df['category'] = large_df['category'].astype('category')  # object → category

print(f"\nOptimized memory usage:")
print(f"  Total: {large_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(large_df.dtypes)

# # Category dtype explained
# print(f"\nCategory 'category' internals:")
# print(f"  Categories: {large_df['category'].cat.categories.tolist()}")
# print(f"  Codes (stored internally): {large_df['category'].cat.codes[:5].tolist()}")
# print(f"  Memory: {large_df['category'].memory_usage(deep=True) / 1024:.1f} KB")

# WHY category? 3 unique strings stored as integers internally.
# 100K strings = ~5MB. 100K integers = ~100KB. 50x savings!

# ANALOGY: Category = library card catalog.
#          Instead of storing full book titles on every shelf marker,
#          you store a number that maps to the title in a master list.


Optimized memory usage:
  Total: 1.05 MB
id              int32
age              int8
income          int32
category     category
is_active        bool
dtype: object


---

# 2. Data Cleaning (Handling NaN, Outliers)

### Topic
Techniques for identifying and resolving missing values, outliers, and inconsistencies.

### Why It Is Related
Real-world data is never clean. Sensors fail, users enter garbage, systems migrate. A model trained on dirty data learns the dirt, not the signal. 'Garbage in, garbage out' is a law of nature.

### How It Works
- **Missing Values**: `isnull()`, `dropna()`, `fillna()`, `interpolate()`
- **Outliers**: Z-score (>3σ), IQR (1.5× rule), visual detection
- **Inconsistencies**: Mixed formats, varying labels, typos

---

### Analogy
🍳 **Data Cleaning = Preparing Ingredients for a Michelin-Star Meal**

- **NaN (missing)**: Salt shaker is empty. Borrow from neighbor (impute), skip the dish (drop), or estimate from other seasonings (model-based).
- **Outliers**: One tomato is rotten. Remove it (drop), cut around the bad part (cap), or blend it anyway (risky).
- **Inconsistencies**: Recipe says '1 cup' but you have metric and imperial cups. Standardize or the cake collapses.

---

## 2.1 Detecting and Handling Missing Values

**RUN THE CELL BELOW** 👇

In [ ]:
# Create a dirty dataset
dirty_df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'age': [25, np.nan, 45, 28, np.nan, 52],
    'income': [45000, 62000, np.nan, 51000, 88000, np.nan],
    'country': ['USA', 'UK', 'USA', np.nan, 'Germany', 'UK'],
    'score': [85, 90, np.nan, 78, 92, np.nan],
})

print("Dirty dataset:")
print(dirty_df)

# Detect missing values
print(f"\nMissing value counts:")
print(dirty_df.isnull().sum())

print(f"\nMissing value percentages:")
print((dirty_df.isnull().mean() * 100).round(2))

# Strategy 1: Drop rows with ANY missing values
dropped = dirty_df.dropna()
print(f"\nAfter dropna(): {len(dropped)} rows remaining (from {len(dirty_df)})")

# Strategy 2: Drop rows where ALL values are missing (none here)
# dropped_all = dirty_df.dropna(how='all')

# Strategy 3: Impute numeric with median
imputed = dirty_df.copy()
imputed['age'] = imputed['age'].fillna(imputed['age'].median())
imputed['income'] = imputed['income'].fillna(imputed['income'].median())
imputed['score'] = imputed['score'].fillna(imputed['score'].mean())

print(f"\nAfter numeric imputation:")
print(imputed[['age', 'income', 'score']])

# Strategy 4: Impute categorical with mode
imputed['country'] = imputed['country'].fillna(imputed['country'].mode()[0])

print(f"\nAfter all imputation:")
print(imputed)
print(f"Missing values remaining: {imputed.isnull().sum().sum()}")

# WHEN to drop vs. impute?
print(f"\nGUIDELINES:")
print(f"  >30% missing in column → DROP column")
print(f"  <5% missing, random → IMPUTE with mean/median/mode")
print(f"  Missing NOT random (e.g., high-income skip salary) → MODEL-BASED impute")

# ANALOGY: Missing data = missing ingredient.
#          Drop = skip the dish.
#          Impute = substitute with something similar.
#          Model-based = ask a chef what they'd use instead.

## 2.2 Outlier Detection & Treatment

**RUN THE CELL BELOW** 👇

In [ ]:
# Create data with outliers
np.random.seed(42)
incomes = np.concatenate([
    np.random.normal(50000, 15000, 100),   # Normal incomes
    [500000, 999999, -5000, 2000000],      # Outliers
])
# incomes

income_df = pd.DataFrame({'income': incomes})
# income_df

# print(f"Income statistics:")
# print(income_df['income'].describe())

# # Method 1: Z-Score
# z_scores = np.abs((income_df['income'] - income_df['income'].mean()) / income_df['income'].std())
# outliers_z = income_df[z_scores > 3]
# print(f"\nZ-Score outliers (>3σ): {len(outliers_z)} rows")
# print(outliers_z.head())

# # Method 2: IQR (Interquartile Range)
Q1 = income_df['income'].quantile(0.25)
Q3 = income_df['income'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


# print(f"\nIQR Method:")
print(f"  Q1: {Q1:.0f}, Q3: {Q3:.0f}, IQR: {IQR:.0f}")
print(f"  Lower bound: {lower_bound:.0f}")
print(f"  Upper bound: {upper_bound:.0f}")

# outliers_iqr = income_df[(income_df['income'] < lower_bound) | (income_df['income'] > upper_bound)]
# print(f"  Outliers: {len(outliers_iqr)} rows")

# # Treatment: Capping (Winsorization)
capped = income_df.copy()
capped['income_capped'] = capped['income'].clip(lower=lower_bound, upper=upper_bound)

# print(f"\nAfter capping:")
# print(capped[['income', 'income_capped']].tail(8))

# # Treatment: Log transform (reduces skewness)
# income_df['income_log'] = np.log1p(income_df['income'] - income_df['income'].min() + 1)

# print(f"\nLog transform reduces outlier impact:")
# print(f"  Original std: {income_df['income'].std():.0f}")
# print(f"  Log std:      {income_df['income_log'].std():.2f}")

# # WHEN are outliers NOT bad?
# print(f"\nIMPORTANT:")
# print(f"  In fraud detection → outliers ARE the signal (keep them!)")
# print(f"  In house prices → outliers skew the model (cap or remove)")

# ANALOGY: Outlier = rotten tomato in a salad.
#          In a salad bowl → remove it (ruins the dish).
#          In a 'find rotten produce' contest → it's the winner (the signal).

15752.749177424084
83042.52455110176


---

# 3. GroupBy Operations

### Topic
The split-apply-combine pattern for aggregating data by categories.

### Why It Is Related
AI datasets are rarely flat. You have user segments, time periods, product categories. GroupBy lets you compute statistics per segment: 'average purchase per age group,' 'conversion rate per channel.' This is how you understand data before modeling it.

### How It Works
1. **Split**: Divide data into groups (`df.groupby('category')`)
2. **Apply**: Run function on each group (sum, mean, custom)
3. **Combine**: Stitch results back into a DataFrame

---

### Analogy
📊 **GroupBy = School Report Card System**

- **Split**: Students divided into classes (9A, 9B, 9C).
- **Apply**: Calculate average math score per class.
- **Combine**: Summary table: 'Class 9A: 85, 9B: 78, 9C: 92.'

---

## 3.1 Basic GroupBy — Split, Apply, Combine

**RUN THE CELL BELOW** 👇

In [29]:
# Create a sales dataset
np.random.seed(42)
sales_df = pd.DataFrame({
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Watch'], 200),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 200),
    'price': np.random.randint(100, 2000, 200),
    'quantity': np.random.randint(1, 10, 200),
    'customer_rating': np.random.randint(1, 6, 200),
})
# sales_df


# Add revenue column
sales_df['revenue'] = sales_df['price'] * sales_df['quantity']

# print("Sample data:")
# print(sales_df.head())

# # Basic GroupBy: average price per product
# print(f"\nAverage price per product:")
# avg_price = sales_df.groupby('product')['price'].mean().round(2)
# print(avg_price)

# # Multiple aggregations
# print(f"\nPrice statistics per product:")
# price_stats = sales_df.groupby('product')['price'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
# print(price_stats)

# # GroupBy multiple columns
print(f"\nRevenue by product AND region:")
product_region = sales_df.groupby(['product', 'region'])['revenue'].sum().round(0)
print(product_region.head(10))

# WHY GroupBy? Before training a recommendation model, you might compute:
# 'average spend per user segment' → these become FEATURES for the model.

# ANALOGY: GroupBy = sorting mail by zip code before delivery.
#          Instead of delivering one by one, you batch by neighborhood.


Revenue by product AND region:
product  region
Laptop   East       41579
         North      49712
         South      36761
         West      106109
Phone    East       56302
         North      54836
         South      49044
         West       51774
Tablet   East       54882
         North      66909
Name: revenue, dtype: int64


## 3.2 Transform vs. Aggregate

**RUN THE CELL BELOW** 👇

In [ ]:
# Aggregate: one row per group
# print("AGGREGATE (one row per group):")
# agg_result = sales_df.groupby('product')['price'].mean()
# print(agg_result)
# print(f"Shape: {agg_result.shape}")

# # Transform: same number of rows as original
# print(f"\nTRANSFORM (same rows, new column):")
# sales_df['price_vs_avg'] = sales_df.groupby('product')['price'].transform(lambda x: x - x.mean())
# sales_df
# print(sales_df[['product', 'price', 'price_vs_avg']].head(10))

# # Transform use case: normalization within group
# sales_df['rating_zscore'] = sales_df.groupby('product')['customer_rating'].transform(
#     lambda x: (x - x.mean()) / x.std()
# )

# print(f"\nZ-score normalization within each product group:")
# print(sales_df[['product', 'customer_rating', 'rating_zscore']].head(10).round(2))

# # WHEN to use which?
# print(f"\nGUIDELINES:")
# print(f"  AGGREGATE → Summary tables, reports, feature engineering")
# print(f"  TRANSFORM → Row-level features, normalization, rankings")

# ANALOGY: Aggregate = class average on report card.
#          Transform = each student's 'deviation from class average.'


TRANSFORM (same rows, new column):


,product,region,price,quantity,customer_rating,revenue,price_vs_avg
0,Tablet,East,727,1,2,727,-279.388889
1,Watch,West,686,8,2,5488,-398.888889
2,Laptop,East,1748,3,4,5244,716.108696
3,Tablet,North,1543,7,2,10801,536.611111
4,Tablet,West,1545,5,4,7725,538.611111
...,...,...,...,...,...,...,...
195,Phone,West,469,6,4,2814,-598.369565
196,Phone,West,735,8,1,5880,-332.369565
197,Watch,South,1229,5,4,6145,144.111111
198,Laptop,West,1793,8,2,14344,761.108696


---

# 4. Merging Private Datasets

### Topic
Combining multiple data sources using joins, merges, and concatenation.

### Why It Is Related
No real dataset is self-contained. A recommendation system needs user profiles (CRM), purchase history (ERP), and product catalogs (PIM). Merging is how you assemble the complete picture.

### How It Works
- **Concat**: Stack DataFrames vertically or horizontally.
- **Merge**: SQL-style joins (inner, left, right, outer, cross).
- **Join**: Merge on index instead of columns.

---

### Analogy
🧩 **Merging = Assembling a Jigsaw Puzzle from Multiple Boxes**

- **Inner Join**: Keep only pieces that fit BOTH puzzles (intersection).
- **Left Join**: Keep your puzzle intact, add matching pieces where they fit.
- **Outer Join**: Combine everything from both puzzles, even unconnected pieces.

---

## 4.1 All Join Types Explained

**RUN THE CELL BELOW** 👇

In [ ]:
# Primary dataset: customers
customers = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005'],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'age': [25, 34, 45, 28, 52],
})

# Secondary dataset: orders (some customers have no orders, some orders have no customer)
orders = pd.DataFrame({
    'order_id': ['O101', 'O102', 'O103', 'O104', 'O105', 'O106'],
    'customer_id': ['C001', 'C001', 'C003', 'C007', 'C002', 'C008'],
    'amount': [120, 340, 560, 90, 230, 450],
})

print("Customers:")
print(customers)
print(f"\nOrders:")
print(orders)

# INNER JOIN: only matching rows in BOTH tables
inner = pd.merge(customers, orders, on='customer_id', how='inner')
print(f"\nINNER JOIN: {len(inner)} rows (customers WITH orders)")
print(inner)

# LEFT JOIN: all customers + matching orders
left = pd.merge(customers, orders, on='customer_id', how='left')
print(f"\nLEFT JOIN: {len(left)} rows (all customers, NaN if no order)")
print(left)

# RIGHT JOIN: all orders + matching customers
right = pd.merge(customers, orders, on='customer_id', how='right')
print(f"\nRIGHT JOIN: {len(right)} rows (all orders, NaN if no customer)")
print(right)

# OUTER JOIN: everything from both
outer = pd.merge(customers, orders, on='customer_id', how='outer')
print(f"\nOUTER JOIN: {len(outer)} rows (all customers + all orders)")
print(outer)

# WHY left join is most common? In AI, you preserve your main dataset
# and enrich it with secondary data. You never want to lose rows from
# your training set just because a lookup table is incomplete.

# ANALOGY: Left join = keeping your original puzzle and adding pieces
#          from a second puzzle where they match. You don't throw away
#          your progress just because the second box is missing a piece.

## 4.2 Merge Validation & Handling Duplicates

**RUN THE CELL BELOW** 👇

In [ ]:
# Danger: duplicate keys cause Cartesian product explosion
customers_dup = pd.DataFrame({
    'customer_id': ['C001', 'C001', 'C002'],  # C001 appears twice!
    'info': ['A', 'B', 'C'],
})

orders_dup = pd.DataFrame({
    'customer_id': ['C001', 'C001', 'C002'],  # C001 appears twice!
    'amount': [100, 200, 300],
})

print("Duplicate keys demo:")
print(f"Customers: {len(customers_dup)} rows")
print(f"Orders: {len(orders_dup)} rows")

merged = pd.merge(customers_dup, orders_dup, on='customer_id')
print(f"\nMerged result: {len(merged)} rows (explosion!)")
print(merged)

# Prevention: validate merge cardinality
print(f"\nPrevention with validate parameter:")
try:
    pd.merge(customers_dup, orders_dup, on='customer_id', validate='one_to_one')
except Exception as e:
    print(f"  Caught error: {type(e).__name__}")
    print(f"  Message: {str(e)[:80]}")

# Best practice: check for duplicates BEFORE merging
print(f"\nBest practice - check duplicates first:")
dup_counts = customers_dup['customer_id'].value_counts()
print(f"  Customer duplicates: {dup_counts[dup_counts > 1].to_dict()}")

# Suffixes for overlapping column names
df1 = pd.DataFrame({'id': [1], 'value': [10]})
df2 = pd.DataFrame({'id': [1], 'value': [20]})
merged_suffix = pd.merge(df1, df2, on='id', suffixes=('_left', '_right'))
print(f"\nSuffixes for overlapping columns:")
print(merged_suffix)

# WHY validate? In production, a bad merge can turn 1M rows into 100M rows,
# crashing your pipeline. Validation catches this before it explodes.

# ANALOGY: Merge validation = a pre-flight checklist.
#          You check fuel, weather, and engine BEFORE takeoff.
#          Catching a duplicate-key explosion before merge is the same.

---

# 5. Pivot Tables & Cross-Tabulation

### Topic
Reshaping data from long format to wide format for analysis and reporting.

### Why It Is Related
Pivot tables reveal patterns hidden in flat data. 'Average revenue by product and region' is a pivot. Feature stores often use pivot-like structures for model inputs.

---

### Analogy
📊 **Pivot Table = Rearranging a Photo Album**

Flat data = photos in chronological order.

Pivot table = photos arranged by 'year' (rows) and 'event type' (columns).

Same photos, different insight.

---

## 5.1 Pivot Tables in Action

**RUN THE CELL BELOW** 👇

In [35]:
# Create a dataset for pivoting
np.random.seed(42)
pivot_df = pd.DataFrame({
    'product': np.random.choice(['Laptop', 'Phone', 'Tablet'], 300),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 300),
    'month': np.random.choice(['Jan', 'Feb', 'Mar', 'Apr'], 300),
    'revenue': np.random.randint(100, 2000, 300),
    'units': np.random.randint(1, 20, 300),
})
# pivot_df
# Pivot: average revenue by product and region
pivot1 = pivot_df.pivot_table(
    values='revenue',
    index='product',
    columns='region',
    aggfunc='mean'
).round(0)
pivot1

# print("Average revenue by product and region:")
# print(pivot1)

# # Pivot: multiple aggregations
# pivot2 = pivot_df.pivot_table(
#     values=['revenue', 'units'],
#     index='product',
#     columns='month',
#     aggfunc={'revenue': 'sum', 'units': 'mean'}
# ).round(0)

# print(f"\nRevenue (sum) and units (mean) by product and month:")
# print(pivot2)

# # Cross-tabulation: frequency counts
# ct = pd.crosstab(pivot_df['product'], pivot_df['region'], margins=True)
# print(f"\nCross-tabulation (count of transactions):")
# print(ct)

# WHY pivot? In AI, you might pivot time-series data:
# 'user_id' (rows) × 'hour_of_day' (columns) × 'click_count' (values)
# This becomes feature matrix for a click-prediction model.

# ANALOGY: Pivot = rearranging Lego bricks.
#          Same bricks, different structure, different purpose.

region,East,North,South,West
product,,,,
Laptop,964.0,965.0,1084.0,1146.0
Phone,1058.0,981.0,908.0,866.0
Tablet,873.0,965.0,903.0,1131.0


---

# 🎯 Week 2 Class 3 Summary

## What We Covered

| Topic | Key Takeaway | Production Pattern |
|-------|-------------|---------------------|
| **DataFrames** | 2D labeled table, columns = Series | Set meaningful index, optimize dtypes |
| **Series** | 1D labeled array with index | .values for NumPy, .loc for labels |
| **Missing Data** | isnull → dropna/fillna/interpolate | >30% missing → drop; <5% → impute |
| **Outliers** | Z-score, IQR, visual detection | Cap for skewed data; remove for normal |
| **GroupBy** | Split → Apply → Combine | Aggregate for summaries; transform for features |
| **Merge** | SQL-style joins on keys | Left join to preserve main data; validate cardinality |
| **Pivot** | Long → wide reshaping | Feature matrix construction for models |

## Learning Checklist

- [ ] I can create a DataFrame and understand the difference from a Python dict.
- [ ] I can detect missing values and choose between drop/impute/remove.
- [ ] I can detect outliers using IQR and Z-score methods.
- [ ] I can standardize inconsistent categorical labels.
- [ ] I can use GroupBy to compute segment statistics.
- [ ] I can merge two datasets using inner, left, and outer joins.
- [ ] I can create pivot tables and correlation matrices.
- [ ] I understand why data cleaning is 80% of an AI engineer's job.

---

*End of Week 2 Class 3 Interactive Lecture*